# Proyecto Productivo IB · Semana 10
## Análisis descriptivo y línea base del problema · Equipo AgroData IC (Grupo 03)  
**Proyecto:** alerta temprana por grados-día para la arañita roja (*Oligonychus punicae*) en palto Hass · Fundo Loma Verde S.A.C.  
**Entrada:** `aranita_roja_analitico_v1.csv` (generado en la Semana 9; 1,462 evaluaciones validadas).  
**Producto:** informe exploratorio v1 con matriz de análisis, medidas, gráficos y 3 hallazgos prudentes.

> Describimos y comparamos. **No** afirmamos causas ni predecimos: eso depende de la evidencia que aún no tenemos.

In [ ]:
import os, shutil, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from IPython.display import display

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 40)
SALIDA = Path('salidas_semana10'); (SALIDA / 'graficos').mkdir(parents=True, exist_ok=True)
ACENTO, GRIS = '#1f6f8b', '#9aa5ab'
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e9eb', 'grid.linewidth': 0.6, 'axes.axisbelow': True})
print('Entorno listo.')

## 1. Cargar el dataset analítico de la Semana 9
Sube `aranita_roja_analitico_v1.csv`.

In [ ]:
ARCHIVO = 'aranita_roja_analitico_v1.csv'
if not os.path.exists(ARCHIVO):
    try:
        from google.colab import files
        up = files.upload()
        if not up: raise FileNotFoundError('No se cargó ningún archivo.')
        nombre = list(up.keys())[0]
        if nombre != ARCHIVO: os.rename(nombre, ARCHIVO)
    except ImportError:
        raise FileNotFoundError(f'Coloca {ARCHIVO} junto al notebook.')
df = pd.read_csv(ARCHIVO, encoding='utf-8-sig', parse_dates=['fecha'])
assert df.shape == (1462, 17), df.shape
assert df['evaluacion_id'].is_unique
assert df['hojas_inf_pct'].between(0, 100).all()
print('Dataset cargado:', df.shape, '| fechas de evaluación:', df['fecha'].nunique())
display(df.head(5))

## 2. Matriz de análisis (antes de calcular)
Cada pregunta define la variable, la medida y el gráfico que la responde. Así evitamos “hacer gráficos porque sí”.

| # | Pregunta | Variables | Medida o comparación | Gráfico |
|---|---|---|---|---|
| 1 | ¿Cómo se distribuye la infestación por evaluación? | `hojas_inf_pct` | Frecuencia, media, mediana, percentiles, RIC | Histograma y boxplot |
| 2 | ¿Cuánto cambia entre módulos? | `modulo`, `hojas_inf_pct` | Media, mediana y n por módulo | Barras (promedio por módulo) |
| 3 | ¿Cómo evoluciona mes a mes? | `mes`, `hojas_inf_pct` | Media, mediana y n por mes | Línea mensual |
| 4 | ¿Qué parte de las evaluaciones supera 10 %, 20 % y 30 %? | `hojas_inf_pct` | Frecuencia relativa sobre el total | Tabla |
| 5 | ¿Se asocia la infestación con el GDA acumulado o la temperatura? | `gda_acum`, `temperatura_oc`, promedio por fecha | Correlación de Spearman | Dispersión por fecha |

**Variable principal:** `hojas_inf_pct`. La `postura_pct` es 0 en la gran mayoría de las evaluaciones, por eso se describe pero no se usa como indicador principal.

## 3. Frecuencias
Unidad: 1 evaluación. Los porcentajes son sobre las 1,462 evaluaciones.

In [ ]:
frec_mod = df['modulo'].value_counts().sort_index().rename('evaluaciones').to_frame()
frec_mod['porcentaje'] = (100 * frec_mod['evaluaciones'] / len(df)).round(1)
display(frec_mod)
frec_mes = df['mes'].value_counts().sort_index().rename('evaluaciones').to_frame()
frec_mes['fechas_evaluadas'] = df.groupby('mes')['fecha'].nunique()
display(frec_mes)
assert frec_mod['evaluaciones'].sum() == len(df) and frec_mes['evaluaciones'].sum() == len(df)
print('El módulo 1 tiene solo', int(frec_mod.loc[1, 'evaluaciones']), 'evaluaciones: sus promedios son poco confiables.')

## 4. Medidas descriptivas
Media, mediana, desviación y percentiles. Como la distribución es asimétrica, la **mediana** describe mejor el caso típico y la **media** refleja los focos altos.

In [ ]:
def resumen(s):
    q = s.quantile([.25, .5, .75, .9, .95])
    return pd.Series({'n': s.count(), 'media': s.mean(), 'mediana': q[.5], 'desv_est': s.std(), 'minimo': s.min(),
                      'p25': q[.25], 'p75': q[.75], 'p90': q[.9], 'p95': q[.95], 'maximo': s.max()})

eval_nivel = pd.DataFrame({'hojas_inf_pct': resumen(df['hojas_inf_pct']), 'postura_pct': resumen(df['postura_pct'])}).T
display(eval_nivel.round(2))

# Clima: se resume por FECHA (cada día de clima se repite en varias evaluaciones; resumir por fila lo ponderaría de más).
por_fecha = df.groupby('fecha').agg(hojas_inf_media=('hojas_inf_pct', 'mean'), n_eval=('evaluacion_id', 'count'),
                                    temperatura_oc=('temperatura_oc', 'first'), humedad_relativa=('humedad_relativa', 'first'),
                                    gda_acum=('gda_acum', 'first'), estado_clima=('estado_calidad_clima', 'first')).reset_index()
assert len(por_fecha) == 50
clima_fecha = pd.DataFrame({v: resumen(por_fecha[v]) for v in ['temperatura_oc', 'humedad_relativa', 'gda_acum']}).T
display(clima_fecha.round(2))

pct_postura0 = 100 * (df['postura_pct'] == 0).mean()
print(f'Evaluaciones con postura = 0: {pct_postura0:.1f} %')
assert abs(pct_postura0 - 90.1) < 0.1 and abs(df['hojas_inf_pct'].mean() - 10.13) < 0.01 and abs(df['hojas_inf_pct'].median() - 3.57) < 0.01

## 5. Distribución de la infestación (pregunta 1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'width_ratios': [1.6, 1]})
ax[0].hist(df['hojas_inf_pct'], bins=np.arange(0, 105, 5), color=ACENTO, edgecolor='white')
ax[0].axvline(df['hojas_inf_pct'].median(), color='#333', lw=1.2, ls='--'); ax[0].axvline(df['hojas_inf_pct'].mean(), color='#c2410c', lw=1.2)
ax[0].text(df['hojas_inf_pct'].median() + 11, ax[0].get_ylim()[1] * 0.92, f"mediana {df['hojas_inf_pct'].median():.1f} %", fontsize=9)
ax[0].text(df['hojas_inf_pct'].mean() + 1, ax[0].get_ylim()[1] * 0.78, f"media {df['hojas_inf_pct'].mean():.1f} %", fontsize=9, color='#c2410c')
ax[0].set_xlabel('% de hojas infestadas por evaluación'); ax[0].set_ylabel('Número de evaluaciones')
ax[0].set_title('La mayoría de evaluaciones tiene poca infestación', loc='left', fontsize=11)
ax[1].boxplot(df['hojas_inf_pct'], vert=True, widths=0.4, patch_artist=True, boxprops=dict(facecolor='#d8e8ee', edgecolor=ACENTO),
              medianprops=dict(color='#333'), flierprops=dict(marker='o', markersize=3, markerfacecolor=GRIS, markeredgecolor='none'))
ax[1].set_xticks([]); ax[1].set_ylabel('% de hojas infestadas'); ax[1].set_title('Valores altos fuera de la caja', loc='left', fontsize=11)
plt.tight_layout(); plt.savefig(SALIDA / 'graficos' / 'g1_distribucion.png', dpi=150); plt.show()

q1, q3 = df['hojas_inf_pct'].quantile([.25, .75]); ric = q3 - q1; limite_sup = q3 + 1.5 * ric
n_atip = int((df['hojas_inf_pct'] > limite_sup).sum())
print(f'RIC = {ric:.2f} | límite superior (Q3 + 1.5·RIC) = {limite_sup:.2f} % | evaluaciones por encima: {n_atip} ({100*n_atip/len(df):.1f} %)')
assert abs(limite_sup - 31.25) < 0.01 and n_atip == 130
print('Estos valores NO son errores: ya pasaron la validación. Son focos de infestación reales y justamente lo que el sistema debe detectar.')

## 6. Comparación por módulo (pregunta 2)

In [ ]:
tab_mod = df.groupby('modulo')['hojas_inf_pct'].agg(n='count', media='mean', mediana='median', p90=lambda s: s.quantile(.9)).round(2)
tab_mod['pct_eval_ge_30'] = (100 * df.assign(g=df['hojas_inf_pct'] >= 30).groupby('modulo')['g'].mean()).round(1)
display(tab_mod)

fig, ax = plt.subplots(figsize=(7.5, 4))
colores = [GRIS if n < 30 else ACENTO for n in tab_mod['n']]
b = ax.bar(tab_mod.index.astype(str), tab_mod['media'], color=colores, width=0.6)
for r, (m, n) in zip(b, zip(tab_mod['media'], tab_mod['n'])):
    ax.text(r.get_x() + r.get_width() / 2, m + 0.3, f'{m:.1f} %\n(n={n})', ha='center', fontsize=8.5)
ax.set_xlabel('Módulo'); ax.set_ylabel('% medio de hojas infestadas'); ax.set_ylim(0, tab_mod['media'].max() * 1.3)
ax.set_title('Promedio de infestación por módulo (gris: pocas evaluaciones)', loc='left', fontsize=11)
plt.tight_layout(); plt.savefig(SALIDA / 'graficos' / 'g2_modulos.png', dpi=150); plt.show()
assert tab_mod.loc[[4, 5], 'media'].min() > tab_mod.loc[[2, 3], 'media'].max() and tab_mod.loc[1, 'n'] == 9

## 7. Evolución mensual (pregunta 3)

In [ ]:
tab_mes = df.groupby('mes')['hojas_inf_pct'].agg(n='count', media='mean', mediana='median').round(2)
tab_mes['fechas'] = df.groupby('mes')['fecha'].nunique()
display(tab_mes)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tab_mes.index, tab_mes['media'], color=ACENTO, lw=2, marker='o', ms=6, label='Media')
ax.plot(tab_mes.index, tab_mes['mediana'], color=GRIS, lw=1.5, marker='o', ms=5, ls='--', label='Mediana')
for x, y in zip(tab_mes.index, tab_mes['media']): ax.text(x, y + 0.6, f'{y:.1f}', ha='center', fontsize=9)
ax.set_ylabel('% de hojas infestadas'); ax.set_xlabel('Mes (2026)'); ax.set_ylim(0, tab_mes['media'].max() * 1.25); ax.legend(frameon=False)
ax.set_title('La infestación media sube de marzo a agosto', loc='left', fontsize=11)
plt.tight_layout(); plt.savefig(SALIDA / 'graficos' / 'g3_evolucion_mensual.png', dpi=150); plt.show()
assert abs(tab_mes.loc['2026-03', 'media'] - 3.01) < 0.01 and abs(tab_mes.loc['2026-08', 'media'] - 16.45) < 0.01

## 8. Frecuencia sobre umbrales (pregunta 4)
Son **cortes exploratorios** para describir la cola de la distribución, no umbrales de decisión validados.

In [ ]:
filas = []
for u in [10, 20, 30]:
    k = int((df['hojas_inf_pct'] >= u).sum())
    filas.append({'corte_%': u, 'evaluaciones': k, 'porcentaje_del_total': round(100 * k / len(df), 1)})
tab_umbral = pd.DataFrame(filas); display(tab_umbral)
assert tab_umbral['evaluaciones'].is_monotonic_decreasing
assert tab_umbral.loc[2, 'porcentaje_del_total'] == 10.1

## 9. Asociación con GDA y temperatura (pregunta 5)
La unidad es la **fecha** (50 fechas): el clima es un valor por día, así que no se correlaciona fila por fila (inflaría la muestra). Se usa Spearman porque la infestación es asimétrica.

In [ ]:
filas = []
for var in ['gda_acum', 'temperatura_oc', 'humedad_relativa']:
    sub = por_fecha[['hojas_inf_media', var]].dropna()
    rho, p = stats.spearmanr(sub[var], sub['hojas_inf_media'])
    filas.append({'variable': var, 'fechas': len(sub), 'rho_spearman': round(rho, 3), 'p_valor': round(p, 3)})
tab_corr = pd.DataFrame(filas); display(tab_corr)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(por_fecha['gda_acum'], por_fecha['hojas_inf_media'], s=20 + 3 * por_fecha['n_eval'], color=ACENTO, alpha=.7, edgecolor='white', linewidth=.8)
r = tab_corr.loc[0]
ax.text(0.03, 0.93, f"Spearman ρ = {r['rho_spearman']:.2f} (p = {r['p_valor']:.2f}, n = {int(r['fechas'])} fechas)", transform=ax.transAxes, fontsize=9.5)
ax.set_xlabel('GDA acumulado del día (°C·día desde 1 de marzo)'); ax.set_ylabel('% medio de hojas infestadas en la fecha')
ax.set_title('El GDA acumulado crece junto con la infestación, pero la relación es débil', loc='left', fontsize=11)
plt.tight_layout(); plt.savefig(SALIDA / 'graficos' / 'g4_gda_vs_infestacion.png', dpi=150); plt.show()
assert abs(tab_corr.loc[0, 'rho_spearman'] - 0.248) < 0.01 and tab_corr.loc[0, 'p_valor'] > 0.05

> **Cuidado al interpretar:** el GDA acumulado **siempre crece con el tiempo** y la infestación también sube de marzo a agosto. Una correlación entre ambos puede deberse solo al paso del tiempo (el calendario), no a un efecto térmico. Por eso aquí **no** la tomamos como evidencia de que el grado-día explique la plaga.

## 10. Tres hallazgos (formato de la docente)
Las cifras de cada hallazgo se comprueban con `assert` en la celda siguiente.

### Hallazgo 1 · La infestación es baja en lo típico y muy alta en pocos casos  
**¿Qué encontramos?** La mediana es 3.6 % de hojas infestadas, pero la media es 10.1 %. El 10.1 % de las evaluaciones llega a 30 % o más y el máximo es 100 %.  
**¿Qué significa?** La distribución es asimétrica: la mayor parte de los puntos está sana y unos pocos focos concentran la infestación. Una alerta basada solo en el promedio se diluiría.  
**¿Qué no podemos afirmar?** Que los valores altos sean errores (pasaron la validación) ni que representen toda la parcela: cada evaluación mira pocas plantas.  
**¿Qué investigaremos después?** Qué umbral usa el Ing. de campo para decidir una aplicación, y si los focos se repiten en las mismas válvulas.

### Hallazgo 2 · La infestación crece de marzo a agosto y no es igual en todos los módulos  
**¿Qué encontramos?** La media mensual pasa de 3.0 % (marzo) a 16.4 % (agosto). Los módulos 4 y 5 tienen medias cercanas a 13 %, frente a 6.8 % y 8.2 % de los módulos 2 y 3.  
**¿Qué significa?** El problema crece en el periodo observado y se concentra más en algunos módulos, así que una alerta debe verse por módulo y no solo para todo el fundo.  
**¿Qué no podemos afirmar?** Que la causa sea el módulo (variedad, edad, riego o manejo) ni que el patrón se repita en otra campaña: hay un solo ciclo. El módulo 1 tiene 9 evaluaciones, así que no se compara.  
**¿Qué investigaremos después?** Diferencias entre módulos en manejo y calendario de evaluación, y si agosto trae más evaluaciones por un cambio de frecuencia de monitoreo (369 evaluaciones, la mayor cantidad del periodo).

### Hallazgo 3 · El GDA acumulado acompaña el crecimiento de la infestación, pero la evidencia es débil  
**¿Qué encontramos?** La correlación entre GDA acumulado e infestación media por fecha es ρ = 0.25 (p = 0.08, 50 fechas): positiva pero no significativa. La temperatura del día casi no varía entre fechas evaluadas.  
**¿Qué significa?** Con estos datos, el grado-día sirve como **contexto térmico** de la campaña, pero no demuestra que anticipe la infestación.  
**¿Qué no podemos afirmar?** Causalidad, ni capacidad de predicción: GDA acumulado y mes crecen juntos; Tbase = 10.34 °C y K = 163.93 GDA vienen de literatura y no están validados en campo.  
**¿Qué investigaremos después?** Validar Tbase y K con el especialista, comparar con una fecha de inicio biológica (biofix) y probar si el GDA ayuda más como señal de periodo de riesgo que como predictor numérico.

In [ ]:
# Cifras citadas en los hallazgos
assert round(df['hojas_inf_pct'].median(), 1) == 3.6 and round(df['hojas_inf_pct'].mean(), 1) == 10.1
assert round(100 * (df['hojas_inf_pct'] >= 30).mean(), 1) == 10.1 and df['hojas_inf_pct'].max() == 100
assert round(df[df['mes'] == '2026-03']['hojas_inf_pct'].mean(), 1) == 3.0 and round(df[df['mes'] == '2026-08']['hojas_inf_pct'].mean(), 1) == 16.4   # media exacta de agosto = 16.447
assert 12.5 < tab_mod.loc[4, 'media'] < 13.6 and 12.5 < tab_mod.loc[5, 'media'] < 13.6
assert 6.5 < tab_mod.loc[2, 'media'] < 8.5 and 6.5 < tab_mod.loc[3, 'media'] < 8.5
assert tab_mes.loc['2026-08', 'n'] == tab_mes['n'].max() == 369
assert round(tab_corr.loc[0, 'rho_spearman'], 2) == 0.25 and round(tab_corr.loc[0, 'p_valor'], 2) == 0.08
print('✓ Todas las cifras de los hallazgos coinciden con los datos.')
print('Rango de temperatura en las fechas evaluadas:', por_fecha['temperatura_oc'].min().round(1), '-', por_fecha['temperatura_oc'].max().round(1), '°C')

## 11. Exportar evidencias
`resultados_estadisticos.csv`, 4 gráficos PNG y `matriz_hallazgos_v1.xlsx`.

In [ ]:
bloques = []
def agrega(nombre, tabla):
    t = tabla.reset_index() if tabla.index.name or isinstance(tabla.index, pd.MultiIndex) else tabla.reset_index(names='fila')
    t.insert(0, 'tabla', nombre); bloques.append(t)
agrega('descriptivas_evaluacion', eval_nivel.round(2)); agrega('descriptivas_clima_por_fecha', clima_fecha.round(2))
agrega('por_modulo', tab_mod); agrega('por_mes', tab_mes); agrega('umbrales_exploratorios', tab_umbral.set_index('corte_%'))
agrega('correlaciones_spearman', tab_corr.set_index('variable'))
resultados = pd.concat(bloques, ignore_index=True)
resultados.to_csv(SALIDA / 'resultados_estadisticos.csv', index=False, encoding='utf-8-sig')

matriz = pd.DataFrame({
 'Hallazgo': ['H1 · Distribución asimétrica', 'H2 · Crecimiento mensual y diferencias entre módulos', 'H3 · GDA y infestación'],
 'Qué encontramos': ['Mediana 3.6 %, media 10.1 %; 10.1 % de evaluaciones ≥ 30 %; máximo 100 %.',
                     'Media mensual de 3.0 % (marzo) a 16.4 % (agosto); módulos 4 y 5 ≈ 13 % vs 6.8 % y 8.2 % en módulos 2 y 3.',
                     'Spearman GDA acumulado vs infestación media por fecha: ρ = 0.25, p = 0.08 (50 fechas).'],
 'Qué significa': ['Pocos focos concentran la infestación; el promedio solo los diluye.',
                   'El problema crece en el periodo y se concentra en algunos módulos: la alerta debe ser por módulo.',
                   'El GDA es contexto térmico; no demuestra capacidad de anticipar la infestación.'],
 'Qué no podemos afirmar': ['Que los valores altos sean errores o que representen toda la parcela.',
                            'La causa de la diferencia entre módulos ni que se repita en otra campaña; el módulo 1 (n=9) no se compara.',
                            'Causalidad ni predicción: GDA y mes crecen juntos; Tbase y K no están validados en campo.'],
 'Qué investigaremos': ['Umbral de decisión del especialista y recurrencia por válvula.',
                        'Manejo por módulo y cambios en la frecuencia de monitoreo.',
                        'Validar Tbase/K, biofix y GDA como señal de periodo de riesgo.'],
 'Evidencia': ['g1_distribucion.png; resultados_estadisticos.csv', 'g2_modulos.png; g3_evolucion_mensual.png', 'g4_gda_vs_infestacion.png']})
with pd.ExcelWriter(SALIDA / 'matriz_hallazgos_v1.xlsx', engine='openpyxl') as w:
    matriz.to_excel(w, index=False, sheet_name='Hallazgos')
    ws = w.sheets['Hallazgos']
    from openpyxl.styles import Alignment, Font, PatternFill
    for col, ancho in zip('ABCDEF', [32, 48, 48, 48, 42, 36]): ws.column_dimensions[col].width = ancho
    for c in ws[1]: c.font = Font(bold=True, color='FFFFFF'); c.fill = PatternFill('solid', fgColor='1F6F8B')
    for fila in ws.iter_rows():
        for c in fila: c.alignment = Alignment(wrap_text=True, vertical='top')
for f in ['resultados_estadisticos.csv', 'matriz_hallazgos_v1.xlsx', 'graficos/g1_distribucion.png', 'graficos/g2_modulos.png',
          'graficos/g3_evolucion_mensual.png', 'graficos/g4_gda_vs_infestacion.png']:
    assert (SALIDA / f).exists(), f
print('✓ Evidencias exportadas en', SALIDA)

# CHECKLIST FINAL
- [x] Matriz de análisis con preguntas, variables, medidas y gráficos
- [x] Frecuencias y medidas descriptivas
- [x] Comparación por grupo (módulo) y evolución en el tiempo (mes)
- [x] Gráficos con título que dice el mensaje, ejes con unidad y n visible
- [x] 3 hallazgos con qué encontramos / qué significa / qué no podemos afirmar / qué investigaremos
- [x] Cifras verificadas con `assert`
- [x] Evidencias: notebook, `resultados_estadisticos.csv`, gráficos y `matriz_hallazgos_v1.xlsx`

In [ ]:
zip_path = shutil.make_archive('semana10_evidencias_AgroDataIC', 'zip', root_dir=SALIDA)
print('Paquete creado:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Fuera de Colab, el ZIP queda guardado en la carpeta de ejecución.')